## TV 테이블 전처리

포인트는 int 컬럼에 공란이 있을 수 있어서 pandas에서는 일반 int 대신 "Int64"를 쓴다. 저장하면 빈 값은 "" 공란으로 나옴.

In [4]:
from pathlib import Path
import pandas as pd
import re

# =========================
# 경로 설정
# =========================
input_path = Path(r"C:/project_skn/project4/4th_project/products/data/preprocessing/TV/lge_tv_all_products.csv")
output_path = Path("ProductTV.csv")

# =========================
# 해상도 화면비 테이블
# =========================
resolution_df = pd.DataFrame({
    "resol_code": ["RES000", "RES001", "RES002"],
    "name": ["4K UHD", "HD", "울트라HD"],
    "width(mm)": [3840, 1366, 3840],
    "height(mm)": [2160, 768, 2160]
})

resolution_map = dict(zip(resolution_df["name"], resolution_df["resol_code"]))

# =========================
# 변환 함수
# =========================
def blank_if_na(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_int(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return int(nums[0])


def clean_max_int(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return max(map(int, nums))


def clean_float(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+(?:\.\d+)?", text)

    if not nums:
        return pd.NA

    return float(nums[0])


def clean_power(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()

    # 예: 0.5↓, 0.5 이하, 0.5아래 화살표 등
    if "0.5" in text:
        return 0.5

    nums = re.findall(r"\d+(?:\.\d+)?", text.replace(",", ""))

    if not nums:
        return pd.NA

    return float(nums[0])


def clean_proficiency(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()

    if "비대상" in text:
        return pd.NA

    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return int(nums[0])


def extract_resolution_name(value):
    if pd.isna(value):
        return ""

    text = str(value).strip()

    # 예: HD (1,366 x 768) -> HD
    name = re.sub(r"\s*\(.*?\)", "", text).strip()

    return name


def split_size(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA, pd.NA, pd.NA

    text = str(value).replace(",", "")
    parts = re.split(r"\s*[xX×]\s*", text)

    if len(parts) < 3:
        return pd.NA, pd.NA, pd.NA

    width = clean_int(parts[0])
    height = clean_int(parts[1])
    depth = clean_int(parts[2])

    return width, height, depth


# =========================
# 원본 읽기
# =========================
df = pd.read_csv(input_path)

# =========================
# 크기 분리
# =========================
size_values = df["스탠드 제외 크기(mm)"].apply(split_size)

df["width(mm)"] = size_values.apply(lambda x: x[0])
df["height(mm)"] = size_values.apply(lambda x: x[1])
df["depth(mm)"] = size_values.apply(lambda x: x[2])

# =========================
# 해상도 코드 매핑
# =========================
df["resolution_name"] = df["해상도"].apply(extract_resolution_name)
df["resol_code"] = df["resolution_name"].map(resolution_map).fillna("")

# =========================
# 새 DataFrame 구성
# =========================
new_df = pd.DataFrame({
    "product_code": [f"TVT{i:04d}" for i in range(len(df))],
    "resol_code": df["resol_code"],
    "name": df["제품명"].apply(blank_if_na),
    "img_link": df["이미지 링크"].apply(blank_if_na),
    "price": df["가격(원)"].apply(clean_int),
    "proficiency_level": df["에너지 소비효율 등급"].apply(clean_proficiency),
    "power_consum(W)": df["소비전력(W)"].apply(clean_power),
    "screen_size(cm)": df["화면사이즈(cm)"].apply(clean_int),
    "display_type": df["디스플레이 종류"].apply(blank_if_na),
    "ref_rate(Hz)": df["주사율(Hz)"].apply(clean_max_int),
    "os_type": df["운영체제"].apply(blank_if_na),
    "speaker_output(W)": df["스피커 출력(W)"].apply(clean_int),
    "width(mm)": df["width(mm)"],
    "height(mm)": df["height(mm)"],
    "depth(mm)": df["depth(mm)"],
    "weight(kg)": df["스탠드 제외 무게(kg)"].apply(clean_float),
    "manual_link": df["제품 사용 설명서"].apply(blank_if_na)
})

# =========================
# 타입 지정
# =========================
str_cols = [
    "product_code", "resol_code", "name", "img_link",
    "display_type", "os_type", "manual_link"
]

int_cols = [
    "price", "proficiency_level", "screen_size(cm)",
    "ref_rate(Hz)", "speaker_output(W)",
    "width(mm)", "height(mm)", "depth(mm)"
]

for col in str_cols:
    new_df[col] = new_df[col].astype("string")

for col in int_cols:
    new_df[col] = new_df[col].astype("Int64")

new_df["power_consum(W)"] = new_df["power_consum(W)"].astype("Float64")
new_df["weight(kg)"] = new_df["weight(kg)"].astype("Float64")

# =========================
# 저장
# =========================
new_df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="")

print(new_df.head())
print(f"저장 완료: {output_path}")

  product_code resol_code                        name  \
0      TVT0000     RES000  LG Micro RGB evo AI (스탠드형)   
1      TVT0001     RES001         LG 일반 LED TV (스탠드형)   
2      TVT0002     RES001         LG 일반 LED TV (스탠드형)   
3      TVT0003     RES001         LG 일반 LED TV (스탠드형)   
4      TVT0004     RES001         LG 일반 LED TV (스탠드형)   

                                            img_link     price  \
0  https://www.lge.co.kr/kr/images/tvs/md10794841...  16500000   
1  https://www.lge.co.kr/kr/images/tvs/md10791858...      <NA>   
2  https://www.lge.co.kr/kr/images/tvs/md10791858...    450000   
3  https://www.lge.co.kr/kr/images/tvs/md10791858...    450000   
4  https://www.lge.co.kr/kr/images/tvs/md10272836...    450000   

   proficiency_level  power_consum(W)  screen_size(cm) display_type  \
0               <NA>              0.5              252    Micro RGB   
1                  1              0.5               80          LED   
2                  1              0.5          

## 소비전력 이상치 검색

In [3]:
import pandas as pd

df = pd.read_csv("C:/project_skn/project4/4th_project/products/data/preprocessing/TV/lge_tv_all_products.csv")
power_counts_df = (
    df["소비전력(W)"]
    .value_counts(dropna=False)
    .reset_index()
)

power_counts_df.columns = ["소비전력(W)", "count"]

print(power_counts_df)

  소비전력(W)  count
0   0.5W↓    186
1    35.4      2
2     NaN      2
3      68      1
